# Modelos no lineales (2): TVP-VAR con volatilidad estocástica

El Threshold VAR del notebook A cambia de golpe entre dos regímenes. La segunda familia de modelos
no lineales cambia **despacio y todo el tiempo**: cada coeficiente es un paseo aleatorio y cada
varianza también. Es el TVP-VAR-SV de Primiceri (2005), escrito en la forma estructural de Chan y
Eisenstat (2018), que es la que estima `MacroPy`:

$$
A_t\,\mathbf{y}_t = \boldsymbol{\mu}_t + B_{1,t}\mathbf{y}_{t-1} + B_{2,t}\mathbf{y}_{t-2} + \boldsymbol{\varepsilon}_t,
\qquad \boldsymbol{\varepsilon}_t \sim N(0, \Sigma_t),\quad \Sigma_t = \mathrm{diag}(e^{h_{1t}}, \ldots, e^{h_{nt}})
$$

$$
\theta_t = \theta_{t-1} + \eta_t, \qquad h_t = h_{t-1} + \zeta_t,
$$

con $A_t$ triangular inferior con unos en la diagonal (la identificación recursiva de la sesión 3,
fecha por fecha) y $\theta_t$ el vector con los coeficientes que se dejan derivar. Apagando
piezas se obtiene una familia:

| modelo | coeficientes | varianzas |
|:-|:-|:-|
| `C` | constantes | constantes |
| `C-SV` | constantes | volatilidad estocástica |
| `TVP-SV` | todos derivan ($\mu_t, B_t, A_t$) | volatilidad estocástica |
| `TVP-R3-SV` | solo los interceptos derivan | volatilidad estocástica |

El sistema es $\mathbf{y}_t = (tot_t,\ y_t,\ \pi_t,\ i_t)'$, la economía pequeña y abierta de la
sesión 3, pero ahora de 1997Q1 a 2026Q2, **con la pandemia adentro**: ningún tratamiento
especial, porque la volatilidad estocástica está para eso.

1. **Estimación** de los cuatro modelos y las volatilidades.
2. **Impulso-respuesta que cambian con el tiempo**: el choque externo y el de política monetaria
   en tres fechas, las superficies completas (interactivas, se pueden rotar) y cómo separar el
   tamaño del choque de su transmisión.
3. **Descomposición de varianza e histórica** con coeficientes y varianzas cambiantes; la FEVD a
   20 trimestres, el largo plazo.
4. **Competencia de modelos sin verosimilitud marginal ni DIC**: qué dice cada uno sobre las
   mismas preguntas, y cómo pronosticaron 2024-2026.

*Tiempo de corrida: dos a tres minutos.*

## 1. Datos y estimación

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from MacroPy import TimeVaryingVAR, generate_series_plot, plot_irf_over_time, set_style
import sys; sys.path.append("..")          # course_data.py vive en la raíz del repositorio
import course_data as cd

set_style()
levels = cd.load()                          # copia congelada de la base del curso
Y = cd.small_open_economy(levels, start="1997-03-01", end="2026-06-01")
names = list(Y.columns)
titles = [cd.LABELS[c] for c in names]
shocks = ["externo", "actividad", "precios", "política"]
print(f"Muestra: {cd.quarter(Y.index[0])} a {cd.quarter(Y.index[-1])} ({len(Y)} trimestres)")
generate_series_plot(Y, series_titles=titles, title="Perú, 1997-2026: la pandemia adentro")

El PBI cae 30 % interanual en 2020Q2 y rebota 42 % un año después. En la sesión 2 esos cuatro
trimestres pedían dummies o el escalamiento de Lenza y Primiceri. Aquí la varianza $e^{h_t}$ puede
subir en 2020 y volver a bajar: la pandemia es un pico de varianza, no un cambio permanente.

`TimeVaryingVAR` estima cualquier miembro de la familia con el argumento `model`. Los estados se
muestrean en bloque con el muestreador de precisión (Chan y Jeliazkov, 2009) y las volatilidades
con la mezcla de Kim, Shephard y Chib (1998). `presample=4` reserva los primeros trimestres como
condiciones iniciales comunes a todos los modelos. Los priors son los del paper
(`TVP_PRIOR_DEFAULTS`): poco informativos sobre los niveles iniciales y conservadores sobre cuánto
puede cambiar cada coeficiente por trimestre.

In [ ]:
specs = ["C", "C-SV", "TVP-SV", "TVP-R3-SV"]
models = {}
for spec in specs:
    m = TimeVaryingVAR(Y, model=spec, lags=2, presample=4, post_draws=15000, burnin=0.2,
                       hor=21, fhor=10, seed=1, progress=False)
    m.sample_posterior()
    models[spec] = m
    print(f"{spec:10s} listo: {m.T} observaciones efectivas, {m.draws['save_idx'].shape[0]} trayectorias de estados guardadas")
tvp = models["TVP-SV"]
tvp.model_summary()

### Volatilidad estocástica

La desviación estándar de cada choque estructural, trimestre a trimestre, con su banda al 68 %.

In [ ]:
tvp.plot_volatility(series_titles=titles, title="TVP-SV: desviación estándar de los choques estructurales")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6.4), constrained_layout=True)
for ax, var, title in zip(axes.ravel(), names, titles):
    for spec, style in (("C-SV", "--"), ("TVP-SV", "-")):
        v = models[spec].volatility_frame()[var]
        ax.plot(v.index, v["median"], style, lw=1.6, label=spec)
    ax.set_title(title)
axes[0, 0].legend()
fig.suptitle("Volatilidad estimada: coeficientes constantes contra coeficientes cambiantes")
plt.show()

Tres lecturas. La volatilidad del PBI se multiplica por diez en 2020-21 (empieza a subir a fines
de 2019 porque la estimación es **suavizada**: usa toda la muestra, incluida información posterior a
cada fecha) y vuelve en 2022 al nivel de la
década anterior: la pandemia es un episodio, no un cambio permanente. La de los términos de
intercambio sube en 2008-09 y otra vez al final de la muestra. La de la tasa de interés cae de
casi nueve puntos en 1998 a cerca de cero desde 2003, el régimen de metas de inflación. Y las dos
especificaciones con volatilidad estocástica cuentan casi la misma historia: la volatilidad es una
propiedad de los datos, no del modelo que se elija para los coeficientes.

## 2. Impulso-respuesta que cambian con el tiempo

Con $A_t^{-1}\Sigma_t^{1/2}$ como factor de Cholesky de cada fecha, `compute_irfs` calcula una
respuesta por trimestre. Guarda la mediana y las bandas para todas las fechas y las extracciones
completas solo para las fechas pedidas.

In [ ]:
fechas = ["2008Q4", "2015Q4", "2024Q4"]
for spec, m in models.items():
    m.compute_irfs(dates=fechas, cred_interval=[0.68, 0.90], hor=21)
tvp.plot_irfs(shock="tot", cred=0.68, series_titles=titles,
              title="TVP-SV: respuesta a un choque de términos de intercambio (mediana y banda al 68 %)")

In [ ]:
plot_irf_over_time(tvp, response="y", shock="tot", horizons=(2, 4, 8), cred=0.68)
tvp.plot_irf_surface(response="y", shock="tot",
                     title="Respuesta del PBI a un choque de términos de intercambio, por fecha")

### Superficies interactivas

La superficie estática resume una sola respuesta. Sánchez y Vassallo (2023) muestran las de todas
las variables ante un mismo choque, por fecha y horizonte. La función siguiente hace lo mismo con
`plotly`: cada superficie se puede **rotar, acercar y leer punto por punto** con el cursor (en
Jupyter, VS Code y Codespaces; no en la vista estática de GitHub). Los colores son los de
`MacroPy`: azul para respuestas negativas y terracota para positivas, centrados en cero.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from matplotlib.colors import to_hex
from MacroPy.style import colormap


def superficies_3d(model, shock, titles, scale=1.0, title=None):
    """Respuesta mediana de cada variable a `shock`, por fecha (eje y) y horizonte (eje x)."""
    j = model.names.index(shock)
    med = model.irf["median"][:, :, :, j] * scale               # (fecha, horizonte, variable)
    fechas = pd.DatetimeIndex(model.irf["dates"])
    anios = fechas.year + (fechas.month - 1) / 12
    horizonte = np.arange(med.shape[1])
    cmap = colormap("diverging")
    escala = [[x, to_hex(cmap(x))] for x in np.linspace(0, 1, 11)]
    n = len(titles)
    cols = 2 if n == 4 else min(n, 3)
    rows = -(-n // cols)
    fig = make_subplots(rows=rows, cols=cols, specs=[[{"type": "scene"}] * cols] * rows,
                        subplot_titles=titles, horizontal_spacing=0.02, vertical_spacing=0.06)
    for k in range(n):
        z = med[:, :, k]
        tope = float(np.abs(z).max()) or 1.0
        fig.add_trace(go.Surface(x=horizonte, y=anios, z=z, colorscale=escala, cmin=-tope, cmax=tope,
                                 showscale=False, opacity=0.95,
                                 hovertemplate="año %{y:.2f}<br>horizonte %{x}<br>respuesta %{z:.2f}<extra></extra>"),
                      row=k // cols + 1, col=k % cols + 1)
    fig.update_scenes(xaxis_title="horizonte", yaxis_title="fecha", zaxis_title="pp",
                      camera=dict(eye=dict(x=1.7, y=-1.5, z=0.9)), aspectmode="manual",
                      aspectratio=dict(x=1, y=1.6, z=0.7))
    fig.update_layout(height=430 * rows, width=1000, title=title, margin=dict(l=0, r=0, t=60, b=0))
    return fig


superficies_3d(tvp, "tot", titles,
               title="TVP-SV: respuestas a un choque de términos de intercambio de una desviación estándar").show()

El choque externo pesó más alrededor de la crisis financiera global y menos después: la
respuesta acumulada del PBI a dos años en 2008Q4 es casi tres veces la de 2024 y cinco veces la de 2015. Es el mismo patrón
que Rodríguez, Vassallo y Castillo (2023) encuentran para la Alianza del Pacífico con datos hasta
2019: la transmisión del precio de exportación alcanzó su máximo en 2009-2010 y luego se moderó.

**Tamaño del choque o transmisión.** Hasta aquí el choque es de **una desviación estándar en cada
fecha**, y esa desviación estándar también cambia: una respuesta mayor en 2008 puede venir de un
choque más grande o de una economía más sensible. Para comparar la transmisión hay que fijar el
mismo tamaño del choque en todas las fechas. Con `irf_1std = 0` el choque es de una unidad (un punto
de crecimiento de los términos de intercambio); se escala a diez puntos para que las cifras se lean
fácil.


In [ ]:
def respuesta_acumulada(model, shock, response, fechas, horizonte=8, scale=1.0):
    """Suma de la respuesta mediana entre el impacto y `horizonte` trimestres, por fecha."""
    t = [[cd.quarter(d) for d in model.yy_dates].index(f) for f in fechas]
    med = model.irf["median"][t, :horizonte + 1, names.index(response), names.index(shock)]
    return med.sum(axis=1) * scale

vol_tot = {cd.quarter(d): v for d, v in tvp.volatility_frame()["tot"]["median"].items()}
tabla = {"desv. estándar del choque externo": [vol_tot[f] for f in fechas],
         "PBI acumulado a 8 trim., choque de 1 d.e.": respuesta_acumulada(tvp, "tot", "y", fechas)}
tvp.irf_1std = 0                                    # choque unitario: mismo tamaño en cada fecha
tvp.compute_irfs(dates=fechas, cred_interval=[0.68, 0.90], hor=21)
tabla["PBI acumulado a 8 trim., choque de +10 pp"] = respuesta_acumulada(tvp, "tot", "y", fechas, scale=10)
tvp.irf_1std = 1                                    # de vuelta al choque de una desviación estándar
tvp.compute_irfs(dates=fechas, cred_interval=[0.68, 0.90], hor=21)
pd.DataFrame(tabla, index=fechas).round(2)

Con el mismo choque de diez puntos en todas las fechas, la respuesta acumulada del PBI a dos años
es 6.1 puntos en 2008Q4, 2.0 en 2015Q4 y 2.7 en 2024Q4. La transmisión sí fue mayor en 2008, entre
dos y tres veces, pero menos de lo que sugería el choque de una desviación estándar (3.9 contra 0.8 y
1.5): el choque externo también era más grande en 2008, con una desviación estándar de 6.3 puntos
frente a 4.1 en 2015 y 5.7 en 2024. **Antes de comparar fechas, hay que fijar el tamaño del choque.**

### La política monetaria

In [ ]:
tvp.plot_irfs(shock="i", cred=0.68, series_titles=titles,
              title="TVP-SV: respuesta a un choque de política monetaria (mediana y banda al 68 %)")

El choque de política se ha vuelto pequeño: su desviación estándar baja de 0.3 puntos en 2008 a
menos de 0.1 en 2024. Con metas de inflación casi todo lo que hace el banco central es sistemático
(sesión 3), y lo que queda como sorpresa mueve poco: la inflación cae unas centésimas y el PBI no
se mueve, con bandas que contienen al cero.

### Los cuatro modelos en una fecha

Misma fecha, mismo choque, cuatro especificaciones. El modelo `C` da la misma respuesta en
cualquier fecha; los otros tres, la de 2024Q4.

In [ ]:
def irf_at(m, date, shock, response):
    t = [cd.quarter(d) for d in m.yy_dates].index(date)
    return m.irf["median"][t, :, names.index(response), names.index(shock)]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), constrained_layout=True)
for ax, (shock, response, title) in zip(axes, [("tot", "y", "PBI ante choque externo"),
                                               ("i", "pi", "Inflación ante choque de política")]):
    for spec, m in models.items():
        ax.plot(irf_at(m, "2024Q4", shock, response), lw=1.7, label=spec)
    ax.axhline(0, color="gray", lw=0.7)
    ax.set_title(f"{title}, 2024Q4")
    ax.set_xlabel("trimestres")
axes[0].legend()
plt.show()

## 3. Descomposición de varianza e histórica

Con coeficientes cambiantes, la FEVD también cambia: `compute_fevd` devuelve la participación de
cada choque por fecha y horizonte. Se reporta a **20 trimestres**, el largo plazo: cuando el
horizonte crece, la FEVD de un VAR estacionario converge a la descomposición de la varianza
incondicional. En un TVP-VAR esa varianza es la del VAR con los parámetros de cada fecha, congelados
hacia adelante. La celda siguiente comprueba que entre 20 y 40 trimestres las participaciones casi
no cambian.

In [ ]:
fevd = tvp.compute_fevd(hor=41)["fevd"]               # (fecha, horizonte, choque, variable)
H = 20                                                  # entrada H - 1: la varianza del error a H pasos
cambio = np.abs(fevd[:, 2 * H - 1] - fevd[:, H - 1]).max()
print(f"Mayor cambio de una participación entre 20 y 40 trimestres, en cualquier fecha: {cambio:.1f} pp")
tvp.plot_fevd(horizon=H, series_titles=titles, shock_titles=shocks,
              title="TVP-SV: descomposición de la varianza del error de pronóstico a 20 trimestres")

In [ ]:
rows = {}
for spec, m in models.items():
    fe = m.compute_fevd(hor=21)["fevd"]                     # (fecha, horizonte, choque, variable)
    for date in fechas:
        t = [cd.quarter(d) for d in m.yy_dates].index(date)
        rows[(spec, date)] = fe[t, H - 1, :, names.index("y")]
pd.DataFrame(rows, index=shocks).T.round(1).rename_axis(["modelo", "fecha"]) \
    .rename(columns=lambda s: f"choque {s}")

La participación del choque externo en la varianza del PBI a 20 trimestres es un número con `C`,
cambia moderadamente con `C-SV` y mucho con `TVP-SV`. Los modelos no discrepan sobre si el
exterior importa, sino sobre **cuánto y cuándo**: con coeficientes cambiantes, el peso del choque
externo en el PBI sube de 44 % en 2008 a 63 % en 2024, contra 8 % fijo en el modelo `C` y 15 a 30 %
con `C-SV`. El peso atribuido al exterior depende mucho de la especificación.

La descomposición histórica reparte cada trimestre observado entre la base y los choques
acumulados, con la identidad contable exacta en cada extracción.

In [ ]:
hd = tvp.compute_hd(plot_hd=True, series_titles=titles, shock_titles=shocks)
print("máx |base + aportes - dato| =", f"{np.abs(hd['baseline'] + hd['shock'].sum(axis=1) - tvp.yy).max():.1e}")

In [ ]:
episodios = {"crisis financiera 2008Q3-2009Q4": ("2008-09-01", "2009-12-01"),
             "pandemia 2020": ("2020-03-01", "2020-12-01"),
             "inflación 2022-2023": ("2022-03-01", "2023-12-01")}
rows = {}
for spec, m in models.items():
    h = m.compute_hd()
    d = pd.DatetimeIndex(h["dates"])
    for name, (a, b) in episodios.items():
        mask = (d >= a) & (d <= b)
        rows[(spec, name)] = {"PBI observado": m.yy[mask, 1].mean(), "base": h["baseline"][mask, 1].mean(),
                              **{f"aporte {s}": h["shock"][mask, k, 1].mean() for k, s in enumerate(shocks)}}
pd.DataFrame(rows).T.round(2).rename_axis(["modelo", "episodio"])

En la pandemia los cuatro modelos coinciden: la caída es casi toda un choque de actividad, y es
enorme. En 2008-09 difieren en cuánto aportó el exterior, entre menos de un punto y casi dos. En
2022-23 los modelos de coeficientes constantes atribuyen casi dos puntos de menor crecimiento a
los choques de precios; el `TVP-SV` solo 0.3, y pone el resto en una base más baja, es decir, en
un crecimiento tendencial que viene cayendo. Las descomposiciones son sensibles a la especificación
justo en los episodios que uno quiere explicar.

## 4. Competencia de modelos sin verosimilitud marginal ni DIC

`MacroPy` calcula ambos criterios (`log_marginal_likelihood`, `dic`), pero son costosos y
`tutorial_tvpvar` ya los cubre. Aquí la comparación es la que hace quien usa los modelos para
algo: ¿cuál pronosticó mejor el periodo reciente? Se reestiman los cuatro con datos hasta 2023Q4
y se evalúan sus densidades predictivas sobre los diez trimestres observados desde entonces.

In [ ]:
origen = "2023-12-01"
Y_pre, actual = Y.loc[:origen], Y.loc["2024-03-01":]
h = len(actual)
forecasts = {}
for spec in specs:
    m = TimeVaryingVAR(Y_pre, model=spec, lags=2, presample=4, post_draws=15000, burnin=0.2,
                       hor=21, fhor=h, seed=1, progress=False)
    m.sample_posterior()
    forecasts[spec] = m.forecast(fhor=h, reps=5)["forecast_draws"]

q = {spec: np.percentile(fc, [16, 50, 84], axis=0) for spec, fc in forecasts.items()}
obs = actual.to_numpy()
rows = {}
for spec in specs:
    lo, med, hi = q[spec]
    for k, var in enumerate(names):
        rows[(spec, var)] = {"RECM de la mediana": np.sqrt(np.mean((med[:, k] - obs[:, k]) ** 2)),
                             "ancho medio de la banda al 68 %": np.mean(hi[:, k] - lo[:, k]),
                             "cobertura al 68 %": np.mean((obs[:, k] >= lo[:, k]) & (obs[:, k] <= hi[:, k]))}
pd.DataFrame(rows).T.round(2).rename_axis(["modelo", "variable"]).unstack("variable").swaplevel(axis=1).sort_index(axis=1)

In [ ]:
dates = actual.index
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), constrained_layout=True)
for ax, var in zip(axes, ["y", "pi", "i"]):
    k = names.index(var)
    hist = Y_pre[var].loc["2022-03-01":]
    ax.plot(hist.index, hist, color="black", lw=1.2, label="observado hasta 2023")
    for spec, color in (("C", "gray"), ("C-SV", "#E78033"), ("TVP-SV", "#1F3D5C")):
        lo, med, hi = q[spec][:, :, k]
        ax.fill_between(dates, lo, hi, color=color, alpha=0.12, lw=0)
        ax.plot(dates, med, color=color, lw=1.8, label=spec)
    ax.plot(dates, actual[var], "o", color="#B0413E", ms=4, label="observado 2024-2026")
    ax.axhline(0, color="gray", lw=0.6, ls=":")
    ax.set_title(f"{cd.LABELS[var]}: bandas al 68 % desde 2023Q4")
axes[0].legend(fontsize=8, loc="upper left")
plt.show()

La lección está en el ancho de las bandas, no en las medianas. El modelo `C` lleva la pandemia en
su varianza para siempre: su banda al 68 % para el PBI es el doble de ancha que la de los modelos
con volatilidad estocástica, que en 2023 ya habían vuelto a una varianza normal. Las medianas son
parecidas y ningún modelo domina en las tres variables: `TVP-R3-SV` acierta más con el PBI y
menos con la inflación y la tasa. La cobertura al 68 % es del 100 % en PBI, inflación y tasa, señal
de que todas las bandas son más anchas de lo necesario, y de 20 a 40 % en los términos de
intercambio: el precio del cobre de 2024-26 sorprendió a todos los modelos. Con diez trimestres
de evaluación no hay un ganador, pero sí un perdedor claro en calibración: la varianza constante.

Es la misma conclusión de Chan y Eisenstat (2018) con datos de Estados Unidos y criterios
formales: casi toda la ganancia viene de la volatilidad estocástica, no de los coeficientes
cambiantes. Primero SV; después, si los datos lo piden, TVP.

## Ideas centrales

* **La volatilidad cambia, y mucho.** La del choque al PBI se multiplica por diez en 2020-21 y
  vuelve; la del choque de tasa cae desde fines de los noventa. Los modelos con volatilidad
  estocástica cuentan la misma historia, y son estimaciones suavizadas: usan toda la muestra.
* **La transmisión también cambia, pero hay que normalizar.** El choque externo pesó más alrededor
  de 2008-09. Con un choque de una desviación estándar en cada fecha cambian a la vez el tamaño del
  choque y la transmisión; con el mismo tamaño, la transmisión de 2008 sigue siendo dos a tres veces
  la de 2015 o 2024.
* **El peso del exterior depende de la especificación.** A 20 trimestres, el choque externo explica
  8 % de la varianza del PBI con `C` y entre 44 y 63 % con `TVP-SV`.
* **Volatilidad estocástica antes que coeficientes cambiantes.** Para el PBI, el ancho medio de la
  banda al 68 % baja de 12.3 a 6.1 puntos al incorporar SV, con medianas parecidas; la cobertura es
  del 100 % en los cuatro modelos. Diez trimestres desde un solo origen no alcanzan para elegir un
  ganador, pero sí muestran que la varianza constante calibra peor.


## Ejercicios

1. Repita la comparación de pronósticos con origen en 2019Q4 y horizonte de ocho trimestres, es
   decir, pronosticando la pandemia. ¿Qué modelo tiene la banda más honesta en 2020Q2?
2. Añada `TVP-R1-SV` (solo $A_t$ deriva) y `TVP-R2-SV` (solo $\mu_t$ y $B_t$ derivan) a la
   familia. ¿Qué parte de la variación en el tiempo del choque externo viene de $A_t$?
3. Calcule `log_marginal_likelihood(n_draws=1000)` para `C-SV` y `TVP-SV` (lea primero el aviso
   de tiempo) y compare con la lectura de la sección 4.
4. Reordene el sistema con la tasa antes de la inflación y repita la superficie de la sección 2.
   ¿Cambia la fecha en que el choque externo pesó más?

## Referencias

- Chan, J.C.C. y Eisenstat, E. (2018). Bayesian model comparison for time-varying parameter VARs with stochastic volatility. *Journal of Applied Econometrics* 33(4), 509-532.
- Cogley, T. y Sargent, T.J. (2005). Drifts and volatilities: monetary policies and outcomes in the post WWII US. *Review of Economic Dynamics* 8(2), 262-302.
- Primiceri, G.E. (2005). Time varying structural vector autoregressions and monetary policy. *Review of Economic Studies* 72(3), 821-852.
- Rodríguez, G., Vassallo, R. y Castillo B., P. (2023). Effects of external shocks on macroeconomic fluctuations in Pacific Alliance countries. *Economic Modelling* 124, 106302.
- Sánchez, W. y Vassallo, R. (2023). Expectativas empresariales e inversión privada en el Perú: un análisis no lineal para el periodo 2003-2019. Consejo Fiscal del Perú, Documento de Investigación 02-2023.